# Claim 1, mục 12: sau RA có còn một phép quay giữa các subject không? (không cần GPU)

**Giả thuyết** (`docs/CLAIM1_RA_BCI2A.md`, mục 6a): RA làm trắng từng subject nên khử được phép trộn kênh A_s,
nhưng chỉ đến một **phép quay** U_s. Phép quay có C(C−1)/2 bậc tự do (3 với 3 kênh, 231 với 22 kênh),
nên phần lệch còn lại sau RA lớn dần theo số kênh.

**Căn chỉnh quay** (Procrustes, bước cuối của RPA): sau RA, xoay từng subject sao cho các tâm lớp
(trong tangent space log-Euclid) khớp với template của các subject nguồn. Subject nguồn dùng nhãn; subject đích
**không dùng nhãn**: tâm lớp lấy theo nhãn giả (tâm template gần nhất), fit phép quay trên session huấn luyện của đích
rồi áp cho session đánh giá.

Với mỗi bộ dữ liệu / tập kênh, MDM và TS-LR được chấm với 5 cấu hình: không RA, RA cố định, RA cố định + quay,
RA theo session, RA theo session + quay (LOSO như trong bài).

| Tập kênh | Mục đích |
|---|---|
| 2a: C3, Cz, C4 | 3 bậc quay |
| 2a: 8 và 14 kênh ngẫu nhiên (3 lần rút mỗi mức) | số bậc quay tăng dần |
| 2a: đủ 22 kênh | 231 bậc quay |
| 2b (3 lưỡng cực), Zhou (14 kênh) | đối chiếu |

**Dự đoán nếu giả thuyết đúng:**
1. `rotation_share` (phần lệch tâm lớp giữa subject sau RA là phép quay) đáng kể, và
2. `extra_rot` (= +quay − không quay) tăng theo số kênh của 2a, lớn nhất ở 22 kênh.

OpenBMI (62 kênh, 54 fold) không chạy ở đây vì quá chậm với phép quay 62×62.

**Thiết lập Kaggle:** Accelerator = **None (CPU)**, Internet = **On**. Không cần Add Input.
Kết quả lưu ở `/kaggle/working/ra_rotation/` và file zip cùng tên.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'moabb>=1.1'], check=True)
import os
os.environ.setdefault('MPLBACKEND', 'Agg')
import mne, moabb
print('mne', mne.__version__, '| moabb', moabb.__version__)


In [ ]:
%%writefile /tmp/ra_data_characteristics.py
"""Claim 1, item 4: measure the data characteristics that could make RA help BCI IV-2a most.

Everything is measured on the signals RA receives (pipeline preprocessing, before RA),
per subject, for BCI IV-2a, BCI IV-2b, Zhou2016 and OpenBMI. No model is trained.

H1  inter-channel correlation / common reference component
    mean |r| between channels; share of signal energy in the common (average) mode,
    i.e. the part that CAR would remove.
H2  out-of-band power (no band-pass on 2a/2b)
    share of 0.5-45 Hz power below 4 Hz, 4-8 Hz, 8-30 Hz and 30-45 Hz; between-subject
    spread of these shares. Zhou2016 is measured twice: pipeline (8-30 Hz) and unfiltered.
H3  quality of the RA reference
    split-half distance of the reference (fit sessions), its condition number, trials per
    channel, and the distance between the fit-session and evaluation-session references.
H4  kind of inter-subject variability
    tangent-space (log-Euclidean) sums of squares split into subject, class,
    subject x class and residual parts, before and after RA. RA removes the subject part;
    it cannot remove the subject x class part.

Used by ra_data_characteristics.ipynb (Kaggle, CPU only), which embeds this file.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.linalg import eigh
from scipy.integrate import trapezoid
from scipy.signal import welch
from sklearn.covariance import OAS

BANDS = {"low_0.5-4": (0.5, 4.0), "theta_4-8": (4.0, 8.0),
         "mubeta_8-30": (8.0, 30.0), "high_30-45": (30.0, 45.0)}
TOTAL_BAND = (0.5, 45.0)          # common to every dataset (OpenBMI is sampled at 100 Hz)
SPLIT_HALF_REPEATS = 10


# ---------------------------------------------------------------- SPD helpers
def spd_power(matrix: np.ndarray, exponent: float) -> np.ndarray:
    values, vectors = eigh(matrix)
    return (vectors * np.maximum(values, 1e-12) ** exponent) @ vectors.T


def spd_log(matrix: np.ndarray) -> np.ndarray:
    values, vectors = eigh(matrix)
    return (vectors * np.log(np.maximum(values, 1e-12))) @ vectors.T


def riemann_mean(covs: np.ndarray, tol: float = 1e-7, max_iter: int = 50) -> np.ndarray:
    """Affine-invariant Riemannian mean (same iteration as the RA used by the model)."""
    mean = covs.mean(axis=0)
    for _ in range(max_iter):
        root, invroot = spd_power(mean, 0.5), spd_power(mean, -0.5)
        tangent = np.mean([spd_log(invroot @ c @ invroot) for c in covs], axis=0)
        if np.linalg.norm(tangent, "fro") < tol:
            break
        values, vectors = eigh(tangent)
        mean = root @ ((vectors * np.exp(values)) @ vectors.T) @ root
        mean = (mean + mean.T) / 2
    return mean


def riemann_distance(a: np.ndarray, b: np.ndarray) -> float:
    values = eigh(b, a, eigvals_only=True)
    return float(np.linalg.norm(np.log(np.maximum(values, 1e-12))))


def oas_covariances(x: np.ndarray) -> np.ndarray:
    return np.stack([OAS().fit(trial.T.astype(np.float64)).covariance_ for trial in x])


def tangent_vectors(covs: np.ndarray) -> np.ndarray:
    """Log-Euclidean vectors: upper triangle of log(C), off-diagonal scaled by sqrt(2)."""
    n = covs.shape[1]
    rows, cols = np.triu_indices(n)
    weight = np.where(rows == cols, 1.0, np.sqrt(2.0))
    return np.stack([spd_log(c)[rows, cols] * weight for c in covs])


# ---------------------------------------------------------------- data container
@dataclass
class SubjectData:
    subject: int
    sessions: list            # [(x [trials, channels, samples] float32, y int), ...] chronological
    fit_sessions: int         # leading sessions used to fit the frozen RA reference


@dataclass
class View:
    name: str                 # e.g. "BCI IV-2a", "Zhou2016 (unfiltered)"
    sfreq: float
    preprocessing: str        # human-readable description
    subjects: list            # [SubjectData]


# ---------------------------------------------------------------- per-subject metrics
def h1_correlation(x: np.ndarray) -> dict:
    """Mean |r| between channels and the common-mode energy share, averaged over trials."""
    xc = x - x.mean(axis=-1, keepdims=True)
    cov = np.einsum("nct,ndt->ncd", xc, xc, optimize=True) / x.shape[-1]
    std = np.sqrt(np.einsum("ncc->nc", cov))
    corr = cov / (std[:, :, None] * std[:, None, :])
    n_ch = x.shape[1]
    off = ~np.eye(n_ch, dtype=bool)
    mean_abs_corr = float(np.abs(corr[:, off]).mean())
    # Energy of the channel average, as a share of total energy: what CAR would remove.
    common = cov.sum(axis=(1, 2)) / n_ch / np.einsum("ncc->n", cov)
    common_share = float(common.mean())
    return {
        "h1_mean_abs_corr": mean_abs_corr,
        "h1_common_mode_share": common_share,
        # 0 = independent equal-variance channels, 1 = identical channels.
        "h1_common_mode_excess": float((common_share - 1 / n_ch) / (1 - 1 / n_ch)),
    }


def mean_psd(x: np.ndarray, sfreq: float) -> tuple[np.ndarray, np.ndarray]:
    nperseg = int(min(x.shape[-1], 2 * sfreq))
    freqs, psd = welch(x.astype(np.float64), fs=sfreq, nperseg=nperseg, axis=-1)
    return freqs, psd.mean(axis=(0, 1))


def h2_band_shares(freqs: np.ndarray, psd: np.ndarray) -> dict:
    def power(lo, hi):
        keep = (freqs >= lo) & (freqs < hi)
        return float(trapezoid(psd[keep], freqs[keep])) if keep.sum() > 1 else 0.0

    total = power(*TOTAL_BAND)
    shares = {f"h2_share_{name}": power(lo, hi) / total for name, (lo, hi) in BANDS.items()}
    shares["h2_share_outside_8-30"] = 1.0 - shares["h2_share_mubeta_8-30"]
    return shares


def h3_reference(subject: SubjectData, rng: np.random.Generator) -> tuple[dict, list]:
    fit_x = np.concatenate([x for x, _ in subject.sessions[:subject.fit_sessions]])
    eval_x = np.concatenate([x for x, _ in subject.sessions[subject.fit_sessions:]])
    n_ch = fit_x.shape[1]
    fit_covs = oas_covariances(fit_x)
    eval_covs = oas_covariances(eval_x)
    reference = riemann_mean(fit_covs)
    halves = []
    for _ in range(SPLIT_HALF_REPEATS):
        order = rng.permutation(len(fit_covs))
        a, b = order[: len(order) // 2], order[len(order) // 2:]
        halves.append(riemann_distance(riemann_mean(fit_covs[a]), riemann_mean(fit_covs[b])))
    values = eigh(reference, eigvals_only=True)
    metrics = {
        "h3_fit_trials": int(len(fit_covs)),
        "h3_trials_per_channel": float(len(fit_covs) / n_ch),
        "h3_split_half_distance": float(np.mean(halves) / np.sqrt(n_ch)),
        "h3_reference_log10_condition": float(np.log10(values.max() / max(values.min(), 1e-12))),
        "h3_session_shift": float(riemann_distance(reference, riemann_mean(eval_covs)) / np.sqrt(n_ch)),
    }
    return metrics, [fit_covs, eval_covs, reference]


# ---------------------------------------------------------------- H4: variance decomposition
def variance_decomposition(vectors: np.ndarray, subjects: np.ndarray, labels: np.ndarray) -> tuple[dict, pd.DataFrame]:
    """Sums of squares of tangent vectors split into subject, class, subject x class, residual."""
    grand = vectors.mean(axis=0)
    total = float(((vectors - grand) ** 2).sum())

    def between(groups):
        ss = 0.0
        for g in np.unique(groups, axis=0):
            mask = np.all(groups == g, axis=1) if groups.ndim == 2 else groups == g
            ss += mask.sum() * float(((vectors[mask].mean(axis=0) - grand) ** 2).sum())
        return ss

    ss_subject = between(subjects)
    ss_class = between(labels)
    ss_cell = between(np.stack([subjects, labels], axis=1))
    ss_interaction = max(ss_cell - ss_subject - ss_class, 0.0)
    summary = {
        "eta2_subject": ss_subject / total,
        "eta2_class": ss_class / total,
        "eta2_subject_x_class": ss_interaction / total,
        "eta2_residual": (total - ss_cell) / total,
    }
    summary["subject_over_interaction"] = summary["eta2_subject"] / max(summary["eta2_subject_x_class"], 1e-12)
    summary["class_over_interaction"] = summary["eta2_class"] / max(summary["eta2_subject_x_class"], 1e-12)

    # Per subject: offset of its mean (removable by RA) vs its class-specific deviation (not removable).
    p = vectors.shape[1]
    class_means = {k: vectors[labels == k].mean(axis=0) for k in np.unique(labels)}
    rows = []
    for s in np.unique(subjects):
        ms = subjects == s
        subject_mean = vectors[ms].mean(axis=0)
        offset = float(np.linalg.norm(subject_mean - grand) / np.sqrt(p))
        dev = [np.sum((vectors[ms & (labels == k)].mean(axis=0) - subject_mean - class_means[k] + grand) ** 2)
               for k in class_means if np.any(ms & (labels == k))]
        interaction = float(np.sqrt(np.mean(dev) / p))
        rows.append({"subject": int(s), "h4_subject_offset": offset,
                     "h4_subject_x_class": interaction,
                     "h4_offset_over_interaction": offset / max(interaction, 1e-12)})
    return summary, pd.DataFrame(rows)


# ---------------------------------------------------------------- one view
def analyse_view(view: View, seed: int = 0, log=print) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    rows, psd_rows = [], []
    vec_raw, vec_shape, vec_ra, subj_ids, labels = [], [], [], [], []
    for sd in view.subjects:
        x_all = np.concatenate([x for x, _ in sd.sessions])
        y_all = np.concatenate([y for _, y in sd.sessions])
        row = {"view": view.name, "subject": sd.subject, "n_channels": x_all.shape[1],
               "n_classes": int(len(np.unique(y_all))), "n_trials": int(len(y_all))}
        row.update(h1_correlation(x_all))
        freqs, psd = mean_psd(x_all, view.sfreq)
        row.update(h2_band_shares(freqs, psd))
        keep = (freqs >= TOTAL_BAND[0]) & (freqs <= TOTAL_BAND[1])
        total = trapezoid(psd[keep], freqs[keep])
        psd_rows += [{"view": view.name, "subject": sd.subject, "freq": float(f), "relative_psd": float(v / total)}
                     for f, v in zip(freqs[keep], psd[keep])]
        h3, (fit_covs, eval_covs, reference) = h3_reference(sd, rng)
        row.update(h3)
        rows.append(row)

        covs = np.concatenate([fit_covs, eval_covs])
        whiten = spd_power(reference, -0.5)
        traces = np.einsum("ncc->n", covs)
        vec_raw.append(tangent_vectors(covs))
        vec_shape.append(tangent_vectors(covs / traces[:, None, None] * covs.shape[1]))
        vec_ra.append(tangent_vectors(np.einsum("ij,njk,kl->nil", whiten, covs, whiten, optimize=True)))
        fit_y = np.concatenate([y for _, y in sd.sessions[:sd.fit_sessions]])
        eval_y = np.concatenate([y for _, y in sd.sessions[sd.fit_sessions:]])
        labels.append(np.concatenate([fit_y, eval_y]))
        subj_ids.append(np.full(len(covs), sd.subject))
        log(f"  {view.name} S{sd.subject:02d}: corr={row['h1_mean_abs_corr']:.2f} "
            f"outside8-30={row['h2_share_outside_8-30']:.2f} split-half={row['h3_split_half_distance']:.3f}")

    subjects = np.concatenate(subj_ids)
    labels = np.concatenate(labels)
    decomposition, per_subject = [], None
    for kind, vectors in (("before RA", vec_raw), ("before RA, trace-normalized", vec_shape),
                          ("after frozen RA", vec_ra)):
        summary, table = variance_decomposition(np.concatenate(vectors), subjects, labels)
        decomposition.append({"view": view.name, "covariance": kind, **summary})
        if kind == "before RA":
            per_subject = table
    metrics = pd.DataFrame(rows).merge(per_subject, on="subject")
    return metrics, pd.DataFrame(decomposition), pd.DataFrame(psd_rows)


# ---------------------------------------------------------------- summaries
SUMMARY_METRICS = [
    "h1_mean_abs_corr", "h1_common_mode_share", "h1_common_mode_excess",
    "h2_share_low_0.5-4", "h2_share_theta_4-8", "h2_share_mubeta_8-30", "h2_share_high_30-45",
    "h2_share_outside_8-30",
    "h3_trials_per_channel", "h3_split_half_distance", "h3_reference_log10_condition", "h3_session_shift",
    "h4_subject_offset", "h4_subject_x_class", "h4_offset_over_interaction",
]


def summarise(metrics: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for view, group in metrics.groupby("view", sort=False):
        row = {"view": view, "n_subjects": len(group), "n_channels": int(group["n_channels"].iloc[0])}
        for metric in SUMMARY_METRICS:
            q1, med, q3 = group[metric].quantile([0.25, 0.5, 0.75])
            row[metric] = f"{med:.3f} [{q1:.3f}-{q3:.3f}]"
        # Between-subject spread of the H1/H2 shares (coefficient of variation).
        for metric in ("h1_common_mode_share", "h2_share_outside_8-30", "h2_share_low_0.5-4"):
            values = group[metric]
            row[f"{metric}_between_subject_cv"] = f"{values.std(ddof=0) / values.mean():.3f}"
        rows.append(row)
    return pd.DataFrame(rows).set_index("view").T


def natural_key(value) -> tuple:
    return tuple(int(part) if part.isdigit() else part for part in re.split(r"(\d+)", str(value)))


def save_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, default=float))


In [ ]:
%%writefile /tmp/ra_interventions.py
"""Claim 1, item 5 (+6): interventions on the data, scored with covariance proxy classifiers.

For one dataset at a time, one data property is switched off (or on) and the RA gain
Delta_RA = acc(with RA) - acc(without RA) is measured again with two proxy classifiers:

MDM    nearest class mean in log-Euclidean tangent space (no learned weights)
TS-LR  tangent-space vectors -> standardisation -> logistic regression

Both use the same tangent vectors (upper triangle of log(C), off-diagonal scaled by sqrt(2)),
so the log-Euclidean distance is the Euclidean distance between vectors.

Protocol (as the paper): leave one subject out. Source subjects contribute their labelled
fit sessions (the training sessions); the target subject is scored on its later sessions.
RA settings, each applied to every subject:
  none         no alignment
  frozen       whitened by the Riemannian mean of the subject's fit sessions
  per_session  every session whitened by its own Riemannian mean
No network is trained. Used by ra_interventions.ipynb (Kaggle, CPU only).
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import ra_data_characteristics as rdc

RA_SETTINGS = ("none", "frozen", "per_session")
CLASSIFIERS = ("MDM", "TS-LR")


@dataclass
class SubjectCovs:
    subject: int
    sessions: list            # [(covs [trials, C, C], y), ...] chronological
    fit_sessions: int


# ---------------------------------------------------------------- signal transforms
def common_average_reference(x: np.ndarray) -> np.ndarray:
    """Subtract the channel mean at every sample (what the OpenBMI pipeline applies)."""
    return x - x.mean(axis=1, keepdims=True)


def to_covariances(view: rdc.View, transform=None, log=print) -> list[SubjectCovs]:
    out = []
    for sd in view.subjects:
        sessions = []
        for x, y in sd.sessions:
            x = transform(x) if transform is not None else x
            sessions.append((rdc.oas_covariances(x), y))
        out.append(SubjectCovs(sd.subject, sessions, sd.fit_sessions))
    log(f"  covariances: {view.name}, {len(out)} subjects, "
        f"{out[0].sessions[0][0].shape[1]} channels{' (transformed)' if transform else ''}")
    return out


# ---------------------------------------------------------------- alignment
def whiten(covs: np.ndarray, reference: np.ndarray) -> np.ndarray:
    w = rdc.spd_power(reference, -0.5)
    return np.einsum("ij,njk,kl->nil", w, covs, w, optimize=True)


def aligned_split(sc: SubjectCovs, setting: str, ref_trials: int | None,
                  rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Tangent vectors and labels of the fit sessions (train) and later sessions (test)."""
    sessions = sc.sessions
    if setting == "frozen":
        fit = np.concatenate([c for c, _ in sessions[:sc.fit_sessions]])
        if ref_trials is not None and ref_trials < len(fit):
            fit = fit[rng.choice(len(fit), ref_trials, replace=False)]
        reference = rdc.riemann_mean(fit)
        sessions = [(whiten(c, reference), y) for c, y in sessions]
    elif setting == "per_session":
        sessions = [(whiten(c, rdc.riemann_mean(c)), y) for c, y in sessions]
    elif setting != "none":
        raise ValueError(setting)
    train, test = sessions[:sc.fit_sessions], sessions[sc.fit_sessions:]
    return (rdc.tangent_vectors(np.concatenate([c for c, _ in train])), np.concatenate([y for _, y in train]),
            rdc.tangent_vectors(np.concatenate([c for c, _ in test])), np.concatenate([y for _, y in test]))


# ---------------------------------------------------------------- classifiers
def mdm_predict(x_train, y_train, x_test):
    classes = np.unique(y_train)
    means = np.stack([x_train[y_train == k].mean(axis=0) for k in classes])
    dist = ((x_test[:, None, :] - means[None]) ** 2).sum(axis=-1)
    return classes[dist.argmin(axis=1)]


def tslr_predict(x_train, y_train, x_test):
    model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))
    return model.fit(x_train, y_train).predict(x_test)


PREDICT = {"MDM": mdm_predict, "TS-LR": tslr_predict}


def loso_accuracy(subjects: list[SubjectCovs], setting: str, ref_trials: int | None = None,
                  repeats: int = 1, seed: int = 0) -> pd.DataFrame:
    """Per-target accuracy (%) of both classifiers, averaged over `repeats` reference draws."""
    rows = []
    for r in range(repeats):
        rng = np.random.default_rng(seed + r)
        parts = {sc.subject: aligned_split(sc, setting, ref_trials, rng) for sc in subjects}
        for target in parts:
            x_tr = np.concatenate([parts[s][0] for s in parts if s != target])
            y_tr = np.concatenate([parts[s][1] for s in parts if s != target])
            _, _, x_te, y_te = parts[target]
            for name in CLASSIFIERS:
                acc = 100.0 * float(np.mean(PREDICT[name](x_tr, y_tr, x_te) == y_te))
                rows.append({"subject": target, "classifier": name, "repeat": r, "acc": acc})
    return (pd.DataFrame(rows).groupby(["subject", "classifier"], as_index=False)["acc"].mean())


def ra_gain_table(dataset: str, intervention: str, subjects: list[SubjectCovs],
                  ref_trials: int | None = None, repeats: int = 1, log=print) -> pd.DataFrame:
    """Accuracy for the three RA settings and the RA gains, per subject and classifier.

    ref_trials limits the trials of the frozen reference only; the other settings are unchanged.
    """
    tables = []
    for setting in RA_SETTINGS:
        reps = repeats if (setting == "frozen" and ref_trials is not None) else 1
        t = loso_accuracy(subjects, setting, ref_trials if setting == "frozen" else None, reps)
        tables.append(t.rename(columns={"acc": f"acc_{setting}"}))
    table = tables[0].merge(tables[1], on=["subject", "classifier"]).merge(tables[2], on=["subject", "classifier"])
    table["gain_frozen"] = table["acc_frozen"] - table["acc_none"]
    table["gain_per_session"] = table["acc_per_session"] - table["acc_none"]
    table.insert(0, "intervention", intervention)
    table.insert(0, "dataset", dataset)
    for name, g in table.groupby("classifier"):
        log(f"  {dataset} | {intervention:<22} | {name:<5}: none {g.acc_none.mean():5.2f}  "
            f"frozen {g.acc_frozen.mean():5.2f} ({g.gain_frozen.mean():+5.2f})  "
            f"per-session {g.acc_per_session.mean():5.2f} ({g.gain_per_session.mean():+5.2f})")
    return table


# ---------------------------------------------------------------- summary
def _p(values: np.ndarray) -> float:
    values = values[np.abs(values) > 1e-9]
    if len(values) == 0:
        return float("nan")
    return float(wilcoxon(values, alternative="two-sided", method="exact" if len(values) <= 25 else "auto").pvalue)


def summarise(results: pd.DataFrame, baseline: dict[str, str]) -> pd.DataFrame:
    """Mean gains per (dataset, intervention, classifier) and the paired change vs the dataset baseline.

    p_gain:   Wilcoxon of the per-subject RA gain against 0.
    p_change: Wilcoxon of the per-subject gain difference (intervention - baseline).
    """
    rows = []
    for (dataset, intervention, clf), g in results.groupby(["dataset", "intervention", "classifier"], sort=False):
        base = results[(results.dataset == dataset) & (results.intervention == baseline[dataset])
                       & (results.classifier == clf)].set_index("subject")
        g = g.set_index("subject")
        row = {"dataset": dataset, "intervention": intervention, "classifier": clf, "n": len(g),
               "acc_none": g.acc_none.mean(), "acc_frozen": g.acc_frozen.mean(),
               "acc_per_session": g.acc_per_session.mean()}
        for kind in ("frozen", "per_session"):
            gain = g[f"gain_{kind}"]
            row[f"gain_{kind}"] = gain.mean()
            row[f"p_gain_{kind}"] = _p(gain.to_numpy())
            if intervention != baseline[dataset]:
                change = (gain - base.loc[g.index, f"gain_{kind}"]).to_numpy()
                row[f"change_vs_baseline_{kind}"] = change.mean()
                row[f"p_change_{kind}"] = _p(change)
        rows.append(row)
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- H3 on OpenBMI: fewer channels
def subset_channels(subjects: list[SubjectCovs], channels: np.ndarray) -> list[SubjectCovs]:
    """The covariance of a channel subset is the corresponding sub-matrix."""
    ix = np.ix_(channels, channels)
    return [SubjectCovs(sc.subject, [(c[:, ix[0], ix[1]], y) for c, y in sc.sessions], sc.fit_sessions)
            for sc in subjects]


def ra_gain_random_channels(dataset: str, intervention: str, subjects: list[SubjectCovs], n_channels: int,
                            draws: int = 5, seed: int = 0, log=print) -> pd.DataFrame:
    """RA gains on `draws` random subsets of `n_channels` channels (same subset for every subject), averaged.

    Fewer channels means more reference trials per channel (H3) without changing the trials.
    """
    rng = np.random.default_rng(seed)
    n_all = subjects[0].sessions[0][0].shape[1]
    tables = []
    for d in range(draws):
        channels = np.sort(rng.choice(n_all, n_channels, replace=False))
        tables.append(ra_gain_table(dataset, intervention, subset_channels(subjects, channels), log=lambda m: None))
        log(f"  {dataset} | {intervention}: draw {d + 1}/{draws} done")
    table = (pd.concat(tables).groupby(["dataset", "intervention", "subject", "classifier"], as_index=False)
             .mean(numeric_only=True))
    for name, g in table.groupby("classifier"):
        log(f"  {dataset} | {intervention:<22} | {name:<5}: none {g.acc_none.mean():5.2f}  "
            f"frozen {g.acc_frozen.mean():5.2f} ({g.gain_frozen.mean():+5.2f})  "
            f"per-session {g.acc_per_session.mean():5.2f} ({g.gain_per_session.mean():+5.2f})")
    return table


In [ ]:
%%writefile /tmp/ra_rotation.py
"""Rotation (Procrustes) alignment after Riemannian alignment (RA).

RA whitens every subject by its own reference, which removes the subject-specific channel
mixing A_s only up to an orthogonal matrix: after RA the subject's data are U_s z, with an
unknown rotation U_s (docs/RA_GIAI_THICH.md, section 3.4). This module estimates a rotation
per subject so that its class means match a common template, as the last step of Riemannian
Procrustes Analysis (Rodrigues et al., 2019), in the log-Euclidean tangent space at the identity.

Rotation acts on covariances by congruence, C -> U C U^T, so log(U C U^T) = U log(C) U^T,
and on signals by x -> U x.

* Source subjects (labelled): generalized Procrustes. The template is the mean of the
  sources' rotated class means; each source is rotated onto it; repeated a few times.
* Target subject (unlabelled): its class means are taken with pseudo-labels (nearest template
  class mean), the rotation is fitted, and pseudo-labels are recomputed; target labels are never used.

The rotation minimizes sum_k w_k ||U A_k U^T - B_k||_F^2 over orthogonal U by Riemannian
gradient descent on the orthogonal group (U <- expm(-t Omega) U, Omega skew-symmetric),
starting at the identity, with backtracking.

Only numpy and scipy. The same file is used by the proxy notebook (analysis/) and by the
network data pipeline (ssm_dan/data/rotation.py).
"""
from __future__ import annotations

import numpy as np
from scipy.linalg import eigh, expm


def spd_log(matrix: np.ndarray) -> np.ndarray:
    values, vectors = eigh(matrix)
    return (vectors * np.log(np.maximum(values, 1e-10))) @ vectors.T


def log_covariances(covs: np.ndarray) -> np.ndarray:
    return np.stack([spd_log(c) for c in covs])


def class_means(logs: np.ndarray, labels: np.ndarray, classes) -> dict:
    return {k: logs[labels == k].mean(axis=0) for k in classes if np.any(labels == k)}


def _cost(rotation, pairs):
    return sum(w * float(np.sum((rotation @ a @ rotation.T - b) ** 2)) for a, b, w in pairs)


def fit_rotation(pairs, iterations: int = 100, tolerance: float = 1e-5) -> np.ndarray:
    """Orthogonal U minimizing sum_k w_k ||U A_k U^T - B_k||^2; pairs = [(A_k, B_k, w_k)]."""
    n = pairs[0][0].shape[0]
    rotation, cost, step = np.eye(n), _cost(np.eye(n), pairs), 1.0
    for _ in range(iterations):
        grad = sum(4.0 * w * (rotation @ a @ rotation.T - b) @ rotation @ a for a, b, w in pairs)
        omega = grad @ rotation.T - rotation @ grad.T          # skew-symmetric
        norm = np.linalg.norm(omega)
        if norm < 1e-12:
            break
        while step > 1e-8:
            candidate = expm(-step * omega / norm) @ rotation
            new_cost = _cost(candidate, pairs)
            if new_cost < cost:
                break
            step /= 2.0
        else:
            break
        improvement = cost - new_cost
        rotation, cost, step = candidate, new_cost, step * 2.0
        if improvement < tolerance * max(cost, 1e-12):
            break
    return rotation


def rotate_logs(logs: np.ndarray, rotation: np.ndarray) -> np.ndarray:
    return np.einsum("ij,njk,lk->nil", rotation, logs, rotation, optimize=True)


def source_template(source_logs: list, source_labels: list, classes, iterations: int = 3):
    """Generalized Procrustes over the labelled sources: (template {class: mean}, [rotation per source])."""
    means = [class_means(L, y, classes) for L, y in zip(source_logs, source_labels)]
    n = source_logs[0].shape[1]
    rotations = [np.eye(n) for _ in means]

    def template_of(rotations):
        return {k: np.mean([u @ m[k] @ u.T for u, m in zip(rotations, means) if k in m], axis=0)
                for k in classes}

    for _ in range(iterations):
        template = template_of(rotations)
        rotations = [fit_rotation([(m[k], template[k], 1.0) for k in m]) for m in means]
    return template_of(rotations), rotations


def predict_nearest(logs: np.ndarray, template: dict) -> np.ndarray:
    keys = list(template)
    means = np.stack([template[k] for k in keys])
    dist = ((logs[:, None] - means[None]) ** 2).sum(axis=(-2, -1))
    return np.asarray(keys)[dist.argmin(axis=1)]


def target_rotation(target_logs: np.ndarray, template: dict, iterations: int = 5) -> np.ndarray:
    """Rotation of an unlabelled subject onto the template, with pseudo-labels."""
    rotation = np.eye(target_logs.shape[1])
    for _ in range(iterations):
        pseudo = predict_nearest(rotate_logs(target_logs, rotation), template)
        if len(np.unique(pseudo)) < 2:
            break
        means = class_means(target_logs, pseudo, template)
        pairs = [(means[k], template[k], float(np.mean(pseudo == k))) for k in means]
        rotation = fit_rotation(pairs)
    return rotation


def misalignment(logs_by_subject: list, labels_by_subject: list, classes) -> dict:
    """Diagnostic with all labels: class-mean mismatch to the template before/after rotation.

    rotation_share = share of the mismatch that rotation removes. Large values mean that after
    RA much of the between-subject difference is a rotation.
    """
    means = [class_means(L, y, classes) for L, y in zip(logs_by_subject, labels_by_subject)]
    plain = {k: np.mean([m[k] for m in means if k in m], axis=0) for k in classes}
    before = float(np.mean([sum(np.sum((m[k] - plain[k]) ** 2) for k in m) for m in means]))
    template, rotations = source_template(logs_by_subject, labels_by_subject, classes)
    after = float(np.mean([sum(np.sum((u @ m[k] @ u.T - template[k]) ** 2) for k in m)
                           for u, m in zip(rotations, means)]))
    return {"mismatch_before": before, "mismatch_after": after,
            "rotation_share": 1.0 - after / max(before, 1e-12)}


In [ ]:
%%writefile /tmp/ra_rotation_proxy.py
"""Claim 1, item 12: does a rotation remain after RA, and does removing it add to the RA gain?

Proxy classifiers (MDM, TS-LR) as in ra_interventions.py, same protocol (leave one subject out,
labelled fit sessions of the sources for training, target scored on its later sessions).
Settings, each with and without rotation alignment (ra_rotation.py):

  none             no alignment
  frozen           RA, reference of the subject's fit sessions
  frozen+rot       frozen, then Procrustes rotation onto the source template
  per_session      RA, one reference per session
  per_session+rot  per_session, then rotation

The target's rotation is fitted on its unlabelled fit sessions with pseudo-labels and applied to
its later sessions; target labels are used only for scoring. Sources are rotated with their labels.

Hypothesis (docs/CLAIM1_RA_BCI2A.md, 6a): RA leaves a subject-specific rotation with C(C-1)/2
degrees of freedom; its effect grows with the channel count, so the extra gain of rotation
(frozen+rot - frozen) should grow from 3 to 22 channels of BCI IV-2a.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

import ra_data_characteristics as rdc
import ra_interventions as rai
import ra_rotation as rr

SETTINGS = ("none", "frozen", "frozen+rot", "per_session", "per_session+rot")
# Diagnostic only (item 15): the target rotation is fitted with the TRUE labels of its fit sessions
# (session T on BCI IV-2a), then applied to its later sessions. Not a valid method.
ORACLE_SETTINGS = ("frozen+oracle", "per_session+oracle")


def tangent_from_logs(logs: np.ndarray) -> np.ndarray:
    n = logs.shape[1]
    rows, cols = np.triu_indices(n)
    weight = np.where(rows == cols, 1.0, np.sqrt(2.0))
    return logs[:, rows, cols] * weight


def aligned_logs(sc: rai.SubjectCovs, base: str):
    """Log-covariances of the fit sessions (train) and later sessions (test) after RA `base`."""
    sessions = sc.sessions
    if base == "frozen":
        reference = rdc.riemann_mean(np.concatenate([c for c, _ in sessions[:sc.fit_sessions]]))
        sessions = [(rai.whiten(c, reference), y) for c, y in sessions]
    elif base == "per_session":
        sessions = [(rai.whiten(c, rdc.riemann_mean(c)), y) for c, y in sessions]
    train, test = sessions[:sc.fit_sessions], sessions[sc.fit_sessions:]
    return (rr.log_covariances(np.concatenate([c for c, _ in train])), np.concatenate([y for _, y in train]),
            rr.log_covariances(np.concatenate([c for c, _ in test])), np.concatenate([y for _, y in test]))


def loso(subjects: list[rai.SubjectCovs], setting: str) -> pd.DataFrame:
    oracle = setting.endswith("+oracle")
    base = setting.replace("+rot", "").replace("+oracle", "")
    rotate = setting.endswith("+rot") or oracle
    parts = {sc.subject: aligned_logs(sc, base) for sc in subjects}
    classes = np.unique(np.concatenate([p[1] for p in parts.values()]))
    rows = []
    for target in parts:
        sources = [s for s in parts if s != target]
        src_logs = [parts[s][0] for s in sources]
        src_y = [parts[s][1] for s in sources]
        tgt_fit, y_fit, tgt_test, y_test = parts[target]
        if rotate:
            template, rotations = rr.source_template(src_logs, src_y, classes)
            src_logs = [rr.rotate_logs(L, u) for L, u in zip(src_logs, rotations)]
            if oracle:
                means = rr.class_means(tgt_fit, y_fit, classes)
                u_target = rr.fit_rotation([(means[k], template[k], 1.0) for k in means])
            else:
                u_target = rr.target_rotation(tgt_fit, template)
            tgt_test = rr.rotate_logs(tgt_test, u_target)
        x_tr = tangent_from_logs(np.concatenate(src_logs))
        y_tr = np.concatenate(src_y)
        x_te = tangent_from_logs(tgt_test)
        for name in rai.CLASSIFIERS:
            acc = 100.0 * float(np.mean(rai.PREDICT[name](x_tr, y_tr, x_te) == y_test))
            rows.append({"subject": target, "classifier": name, "acc": acc})
    return pd.DataFrame(rows)


def rotation_table(dataset: str, view: str, subjects: list[rai.SubjectCovs], log=print,
                   oracle: bool = False) -> pd.DataFrame:
    table = None
    for setting in SETTINGS + (ORACLE_SETTINGS if oracle else ()):
        t = loso(subjects, setting).rename(columns={"acc": f"acc_{setting}"})
        table = t if table is None else table.merge(t, on=["subject", "classifier"])
    for base in ("frozen", "per_session"):
        table[f"gain_{base}"] = table[f"acc_{base}"] - table["acc_none"]
        table[f"gain_{base}+rot"] = table[f"acc_{base}+rot"] - table["acc_none"]
        table[f"extra_rot_{base}"] = table[f"acc_{base}+rot"] - table[f"acc_{base}"]
        if f"acc_{base}+oracle" in table:
            table[f"extra_oracle_{base}"] = table[f"acc_{base}+oracle"] - table[f"acc_{base}"]
    table.insert(0, "n_channels", subjects[0].sessions[0][0].shape[1])
    table.insert(0, "view", view)
    table.insert(0, "dataset", dataset)
    for name, g in table.groupby("classifier"):
        log(f"  {dataset} | {view:<18} | {name:<5}: none {g.acc_none.mean():5.2f} | "
            f"frozen {g['acc_frozen'].mean():5.2f} +rot {g['acc_frozen+rot'].mean():5.2f} "
            f"({g.extra_rot_frozen.mean():+5.2f}) | per-session {g['acc_per_session'].mean():5.2f} "
            f"+rot {g['acc_per_session+rot'].mean():5.2f} ({g.extra_rot_per_session.mean():+5.2f})"
            + (f" | oracle: frozen {g.extra_oracle_frozen.mean():+5.2f}, per-session "
               f"{g.extra_oracle_per_session.mean():+5.2f}" if "extra_oracle_frozen" in g else ""))
    return table


def rotation_diagnostic(dataset: str, view: str, subjects: list[rai.SubjectCovs]) -> dict:
    """Share of the class-mean mismatch between subjects (after frozen RA) that is a rotation.

    Uses the labels of every subject: a description of the data, not a classifier.
    """
    parts = [aligned_logs(sc, "frozen") for sc in subjects]
    logs = [np.concatenate([p[0], p[2]]) for p in parts]
    labels = [np.concatenate([p[1], p[3]]) for p in parts]
    classes = np.unique(np.concatenate(labels))
    out = rr.misalignment(logs, labels, classes)
    return {"dataset": dataset, "view": view, "n_channels": subjects[0].sessions[0][0].shape[1], **out}


def _p(values) -> float:
    values = np.asarray(values)
    values = values[np.abs(values) > 1e-9]
    if len(values) == 0:
        return float("nan")
    return float(wilcoxon(values, method="exact" if len(values) <= 25 else "auto").pvalue)


def summarise(results: pd.DataFrame) -> pd.DataFrame:
    rows = []
    keys = ["dataset", "view", "n_channels", "classifier"]
    for key, g in results.groupby(keys, sort=False):
        # Random channel subsets: average the draws per subject first.
        g = g.groupby("subject").mean(numeric_only=True)
        row = dict(zip(keys, key), n=len(g))
        for setting in SETTINGS + ORACLE_SETTINGS:
            if f"acc_{setting}" in g:
                row[f"acc_{setting}"] = g[f"acc_{setting}"].mean()
        for base in ("frozen", "per_session"):
            row[f"gain_{base}"] = g[f"gain_{base}"].mean()
            row[f"gain_{base}+rot"] = g[f"gain_{base}+rot"].mean()
            row[f"extra_rot_{base}"] = g[f"extra_rot_{base}"].mean()
            row[f"p_extra_rot_{base}"] = _p(g[f"extra_rot_{base}"])
            if f"extra_oracle_{base}" in g:
                row[f"extra_oracle_{base}"] = g[f"extra_oracle_{base}"].mean()
                row[f"p_extra_oracle_{base}"] = _p(g[f"extra_oracle_{base}"])
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
import json, shutil, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import mne
from moabb import datasets as moabb_datasets
from moabb.utils import set_download_dir

sys.path.insert(0, '/tmp')
import ra_data_characteristics as rdc

mne.set_log_level('ERROR')
DATA_DIR = Path('/tmp/mne_data'); DATA_DIR.mkdir(parents=True, exist_ok=True)
set_download_dir(str(DATA_DIR))
OUT = Path('/kaggle/working/ra_rotation'); OUT.mkdir(parents=True, exist_ok=True)

# Same windows and filters as the training pipeline (configs/hada_tcformer.yaml):
# BCI IV-2a full MOABB interval, no filter; BCI IV-2b interval minus its last 0.5 s, no filter;
# Zhou2016 8-30 Hz and 0.5-5.0 s after the cue. fit = leading sessions used for the frozen RA reference.
SPECS = {
    'BCI IV-2a': dict(classes=('BNCI2014_001', 'BNCI2014001'), start=0.0, duration=4.0, fit=1, sessions=2),
    'BCI IV-2b': dict(classes=('BNCI2014_004', 'BNCI2014004'), start=0.0, duration=4.0, fit=3, sessions=5),
    'Zhou2016': dict(classes=('Zhou2016',), start=0.5, duration=4.5, fit=2, sessions=3),
}


def moabb_dataset(names):
    for name in names:
        if hasattr(moabb_datasets, name):
            return getattr(moabb_datasets, name)()
    raise ImportError(f'none of {names} in this MOABB version')


def epochs_from_raw(raw, dataset, start, duration, band):
    event_id = dataset.event_id
    stim = [ch for ch, kind in zip(raw.ch_names, raw.get_channel_types()) if kind == 'stim']
    if stim:
        events = mne.find_events(raw, stim_channel=stim[0], shortest_event=1, verbose=False)
    else:
        events, _ = mne.events_from_annotations(raw, event_id=event_id, verbose=False)
    codes = sorted(event_id.values())
    events = events[np.isin(events[:, 2], codes)]
    eeg = raw.copy().pick('eeg').load_data(verbose=False)
    eeg.apply_function(lambda values: values * 1e6, channel_wise=False)   # V -> uV, as the pipeline
    if band is not None:
        eeg.filter(band[0], band[1], verbose=False)
    sfreq = eeg.info['sfreq']
    tmin = float(dataset.interval[0]) + start
    present = {name: code for name, code in event_id.items() if code in set(events[:, 2])}
    epochs = mne.Epochs(eeg, events, event_id=present, tmin=tmin, tmax=tmin + duration - 1.0 / sfreq,
                        baseline=None, preload=True, verbose=False)
    y = np.searchsorted(codes, epochs.events[:, 2]).astype(np.int64)
    return epochs.get_data().astype(np.float32), y, sfreq


def load_moabb_views(name, spec, bands):
    """One View per band setting; raw files are read once per subject."""
    dataset = moabb_dataset(spec['classes'])
    subjects = {label: [] for label in bands}
    sfreq = None
    for subject in dataset.subject_list:
        started = time.time()
        data = dataset.get_data(subjects=[subject])[subject]
        session_names = sorted(data, key=rdc.natural_key)
        if len(session_names) != spec['sessions']:
            raise RuntimeError(f'{name} S{subject}: {len(session_names)} sessions, expected {spec["sessions"]}')
        for label, band in bands.items():
            sessions = []
            for session in session_names:
                parts = [epochs_from_raw(data[session][run], dataset, spec['start'], spec['duration'], band)
                         for run in sorted(data[session], key=rdc.natural_key)]
                sfreq = parts[0][2]
                sessions.append((np.concatenate([p[0] for p in parts]), np.concatenate([p[1] for p in parts])))
            subjects[label].append(rdc.SubjectData(subject, sessions, spec['fit']))
        shapes = [s[0].shape for s in subjects[next(iter(bands))][-1].sessions]
        print(f'{name} S{subject:02d}: sessions {shapes} ({time.time() - started:.0f} s)', flush=True)
    return [rdc.View(label, sfreq, desc, subjects[label]) for label, desc in
            ((label, f'{spec["start"]}-{spec["start"] + spec["duration"]} s after interval start, band {band}')
             for label, band in bands.items())]


def load_openbmi_views():
    """OpenBMI caches attached under /kaggle/input, grouped by their preprocessing signature."""
    groups = {}
    for path in sorted(Path('/kaggle/input').rglob('openbmi_S*.npz')):
        with np.load(path, allow_pickle=False) as data:
            signature = json.loads(str(data['signature']))
        car = 'CAR' if signature.get('common_average_reference', True) else 'no CAR'
        groups.setdefault((car, signature['sfreq'], signature['low_cut'], signature['high_cut']), {}) \
              .setdefault(int(path.stem[-2:]), path)
    views = []
    for (car, sfreq, low, high), files in sorted(groups.items()):
        if len(files) != 54:
            print(f'OpenBMI {car}: only {len(files)}/54 subjects attached, used as is', flush=True)
        subjects = []
        for subject in sorted(files):
            with np.load(files[subject], allow_pickle=False) as data:
                subjects.append(rdc.SubjectData(subject, [(data['X1'], data['y1']), (data['X2'], data['y2'])], 1))
        views.append(rdc.View(f'OpenBMI ({car}, {low:g}-{high:g} Hz)', float(sfreq),
                              f'{car}, {low}-{high} Hz, {sfreq} Hz, 0-4 s', subjects))
        print(f'OpenBMI {car}: {len(subjects)} subjects', flush=True)
    if not views:
        print('No OpenBMI cache attached: add openbmi-dataset-cuong (and openbmi-nocar-cuong) via Add Input.')
    return views

import ra_interventions as rai
import ra_rotation_proxy as rrp


In [ ]:
log = lambda msg: print(msg, flush=True)
results, diagnostics = [], []

def run(dataset, view, covs):
    started = time.time()
    results.append(rrp.rotation_table(dataset, view, covs, log=log))
    diagnostics.append(rrp.rotation_diagnostic(dataset, view, covs))
    log(f"    rotation_share {diagnostics[-1]['rotation_share']:.3f} ({(time.time() - started) / 60:.1f} min)")
    pd.concat(results, ignore_index=True).to_csv(OUT / 'subject_results.csv', index=False)
    pd.DataFrame(diagnostics).to_csv(OUT / 'rotation_diagnostic.csv', index=False)

# ---- BCI IV-2a: channel names from MOABB (EEG channels in the order the loader keeps)
dataset_2a = moabb_dataset(SPECS['BCI IV-2a']['classes'])
first = dataset_2a.get_data(subjects=[1])[1]
session = sorted(first, key=rdc.natural_key)[0]
raw = first[session][sorted(first[session], key=rdc.natural_key)[0]]
names = raw.copy().pick('eeg').ch_names
motor = [names.index(c) for c in ('C3', 'Cz', 'C4')]
log(f'2a channels ({len(names)}): {names}; C3/Cz/C4 at {motor}')

a_raw, = load_moabb_views('BCI IV-2a', SPECS['BCI IV-2a'], {'raw': None})
covs_2a = rai.to_covariances(a_raw, log=log)
del a_raw
run('BCI IV-2a', 'C3 Cz C4', rai.subset_channels(covs_2a, np.array(motor)))
rng = np.random.default_rng(0)
for n in (8, 14):
    for draw in range(3):
        channels = np.sort(rng.choice(len(names), n, replace=False))
        run('BCI IV-2a', f'{n} random', rai.subset_channels(covs_2a, channels))
run('BCI IV-2a', 'all 22', covs_2a)

# ---- BCI IV-2b and Zhou2016 (pipeline preprocessing)
b_raw, = load_moabb_views('BCI IV-2b', SPECS['BCI IV-2b'], {'raw': None})
run('BCI IV-2b', 'all 3', rai.to_covariances(b_raw, log=log))
del b_raw
z_pipe, = load_moabb_views('Zhou2016', SPECS['Zhou2016'], {'8-30': (8.0, 30.0)})
run('Zhou2016', 'all 14', rai.to_covariances(z_pipe, log=log))
del z_pipe

results = pd.concat(results, ignore_index=True)
diagnostics = pd.DataFrame(diagnostics)


In [ ]:
summary = rrp.summarise(results)
summary.to_csv(OUT / 'summary.csv', index=False)
diag = diagnostics.groupby(['dataset', 'view', 'n_channels'], sort=False, as_index=False).mean(numeric_only=True)
diag.to_csv(OUT / 'rotation_diagnostic_summary.csv', index=False)
pd.set_option('display.width', 250, 'display.max_columns', 40)
display(diag.round(3))
cols = ['dataset', 'view', 'n_channels', 'n', 'acc_none', 'acc_frozen', 'acc_frozen+rot', 'extra_rot_frozen',
        'p_extra_rot_frozen', 'acc_per_session', 'acc_per_session+rot', 'extra_rot_per_session', 'p_extra_rot_per_session']
for clf in rai.CLASSIFIERS:
    print(f'===== {clf}')
    display(summary[summary.classifier == clf][cols].round(3))


In [ ]:
import matplotlib.pyplot as plt
two_a = summary[summary.dataset == 'BCI IV-2a'].sort_values('n_channels')
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for clf in rai.CLASSIFIERS:
    s = two_a[two_a.classifier == clf]
    axes[0].plot(s.n_channels, s.extra_rot_frozen, 'o-', label=f'{clf}, fixed RA')
    axes[0].plot(s.n_channels, s.extra_rot_per_session, 's--', label=f'{clf}, per-session RA')
    axes[1].plot(s.n_channels, s.gain_frozen, 'o-', label=f'{clf}: RA')
    axes[1].plot(s.n_channels, s['gain_frozen+rot'], 's--', label=f'{clf}: RA + rotation')
axes[0].axhline(0, color='0.4', lw=0.8)
axes[0].set(xlabel='channels (BCI IV-2a)', ylabel='accuracy points', title='Extra gain of rotation after RA')
axes[1].set(xlabel='channels (BCI IV-2a)', ylabel='accuracy points', title='Gain over no alignment (fixed RA)')
d = diag[diag.dataset == 'BCI IV-2a'].sort_values('n_channels')
axes[2].plot(d.n_channels, d.rotation_share, 'o-', label='BCI IV-2a')
for _, r in diag[diag.dataset != 'BCI IV-2a'].iterrows():
    axes[2].plot(r.n_channels, r.rotation_share, 'x', ms=10, label=r.dataset)
axes[2].set(xlabel='channels', ylabel='share', ylim=(0, 1), title='Class-mean mismatch after RA that is a rotation')
for ax in axes:
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / 'rotation_vs_channels.png', dpi=150); plt.show()

shutil.make_archive(str(OUT), 'zip', root_dir=OUT)
print('Saved', sorted(p.name for p in OUT.iterdir()), 'and', f'{OUT}.zip')


## Cách đọc kết quả

| Nếu thấy | Thì |
|---|---|
| `extra_rot` của 2a tăng theo số kênh, dương và có ý nghĩa ở 22 kênh | Ủng hộ giả thuyết phép quay; làm tiếp mục 13 (mạng) |
| `extra_rot` ≈ 0 ở mọi số kênh | Phép quay còn lại không ảnh hưởng bộ phân loại; giả thuyết yếu đi |
| `rotation_share` lớn nhưng `extra_rot` ≈ 0 | Có phép quay, nhưng không ước lượng được nó cho đích khi không có nhãn (nhãn giả kém) |
| `extra_rot` âm | Nhãn giả của đích sai, phép quay xoay nhầm lớp |

Gửi lại file zip (`summary.csv`, `rotation_diagnostic_summary.csv`, `subject_results.csv`, hình).
